# **Gemma-2B Customer Support Chatbot: QLoRA Fine-Tuning**

## 1. Install Required Libraries

In [1]:
!pip install -q -U "pyarrow<24" transformers datasets accelerate peft trl bitsandbytes sentencepiece huggingface_hub evaluate rouge_score

In [2]:
import os
import torch
from google.colab import userdata

from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)

##  Configure Hugging Face Authentication

In [3]:
HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN if HF_TOKEN else ""

if HF_TOKEN:
    print("HF_TOKEN is set")
else:
    print("HF_TOKEN is not set")


HF_TOKEN is set


## Load the Bitext Customer Support Dataset

In [4]:
DATASET_NAME = "bitext/Bitext-customer-support-llm-chatbot-training-dataset"

dataset = load_dataset(DATASET_NAME)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 26872
    })
})


In [5]:
print("\nTotal Examples:", len(dataset["train"]))
print("\nTraining columns:", dataset["train"].column_names)
print("\nFirst example:")
print(dataset["train"][0])


Total Examples: 26872

Training columns: ['flags', 'instruction', 'category', 'intent', 'response']

First example:
{'flags': 'B', 'instruction': 'question about cancelling order {{Order Number}}', 'category': 'ORDER', 'intent': 'cancel_order', 'response': "I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you."}


## Split Dataset into Training and Validation Sets

In [6]:
split_dataset = dataset["train"].train_test_split(
    test_size=0.10,
    seed=42
)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print("Training examples:", len(train_dataset))
print("Validation examples:", len(eval_dataset))

Training examples: 24184
Validation examples: 2688


In [7]:
train_dataset = train_dataset.select(range(3000))
eval_dataset = eval_dataset.select(range(300))

print("Training samples:", len(train_dataset))
print("Validation samples:", len(eval_dataset))

Training samples: 3000
Validation samples: 300


## Format Dataset Instruction & Response Pairs

In [8]:
def format_chat(example):
    return {
        "text": f"""### Instruction:
{example["instruction"]}

### Response:
{example["response"]}"""
    }

train_dataset = train_dataset.map(format_chat)
eval_dataset = eval_dataset.map(format_chat)

print(train_dataset[0]["text"])

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

### Instruction:
could uhelp me close a freemium account

### Response:
Thank you for your message to us for assistance with closing your {{Account Category}} account. I understand that you're looking to terminate your account and I'm here to guide you through the process. To cancel your {{Account Category}} account, please follow the steps outlined on our website or get in touch with our customer support team who will be happy to help you with the account closure. Rest assured, we're committed to providing you with the support you need to successfully close your {{Account Category}} account.


## Select the Pretrained Gemma-2B Model

In [9]:
model_name = "google/gemma-2b"

print("Model:", model_name)

Model: google/gemma-2b


##  Configure 4-Bit Quantization with BitsAndBytes

In [10]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("4-bit quantization configuration ready!")

4-bit quantization configuration ready!


## Load the Quantized Gemma-2B Model

In [11]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN
)

print("Gemma-2B loaded successfully!")
print("Model device:", model.device)

[transformers] We found `hidden_act="gelu"` in this Gemma config. This is a legacy value of the official releases but it is meant to target the tanh approximation. Setting `hidden_act="gelu_pytorch_tanh"` instead.


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

Gemma-2B loaded successfully!
Model device: cuda:0


## Load the Gemma-2B Tokenizer

In [12]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=HF_TOKEN
)

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


In [13]:
tokenizer.pad_token = tokenizer.eos_token

print("Pad token:", tokenizer.pad_token)

Pad token: <eos>


## Prepare Model for QLoRA Training

In [14]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

print("Model prepared for QLoRA training!")

Model prepared for QLoRA training!


## Configure LoRA Adapters

In [15]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,686,400 || all params: 2,509,858,816 || trainable%: 0.1469


## Configure Training Hyperparameters

In [16]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gemma-2b-bitext",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    eval_strategy="steps",
    save_total_limit=1,
    report_to="none",
    optim="paged_adamw_8bit",
)

print("Training arguments configured.")

Training arguments configured.


## Initialize the Supervised Fine-Tuning Trainer

In [17]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print("SFT Trainer created successfully!")

Adding EOS to train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

SFT Trainer created successfully!


## Fine-Tune Gemma-2B on the Customer Support Dataset

In [18]:
train_result = trainer.train()

print("Fine-tuning completed!")
print(train_result.metrics)

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
100,1.059338,1.067406,1.054645,0.711923,119011.000000
200,0.954225,0.961014,0.960535,0.730787,237848.000000
300,0.952173,0.908518,0.946093,0.740221,357736.000000
375,0.887136,0.898214,0.932421,0.741062,444294.000000


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

Fine-tuning completed!
{'train_runtime': 1342.8166, 'train_samples_per_second': 2.234, 'train_steps_per_second': 0.279, 'total_flos': 7837079779885056.0, 'train_loss': 1.0536345876057942, 'epoch': 1.0}


## Save the Fine-Tuned Model and Tokenizer

In [19]:
OUTPUT_DIR = "./gemma-support-qlora-final"

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Fine-tuned adapter and tokenizer saved to: {OUTPUT_DIR}")

Fine-tuned adapter and tokenizer saved to: ./gemma-support-qlora-final


In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
import os
import shutil

DRIVE_DIR = "/content/drive/MyDrive/Gemma_Support_QLoRA"

os.makedirs(DRIVE_DIR, exist_ok=True)

shutil.copytree(
    OUTPUT_DIR,
    DRIVE_DIR,
    dirs_exist_ok=True
)

print("Model saved successfully to Google Drive!")
print("Location:", DRIVE_DIR)
print("Files:", os.listdir(DRIVE_DIR))

Model saved successfully to Google Drive!
Location: /content/drive/MyDrive/Gemma_Support_QLoRA
Files: ['adapter_config.json', 'tokenizer.json', 'tokenizer_config.json', 'adapter_model.safetensors', 'training_args.bin', 'README.md']


## Reload the Base Model and Fine-Tuned LoRA Adapter

In [22]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME = "google/gemma-2b"
ADAPTER_PATH = "/content/drive/MyDrive/Gemma_Support_QLoRA"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

# Load the base model in 4-bit to fit the T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN
)

# Attach your trained LoRA adapter
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

print("Fine-tuned Gemma-2B loaded successfully!")

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

Fine-tuned Gemma-2B loaded successfully!


## Define the Customer Support Response Generation Function

In [23]:
def generate_response(question):
    prompt = f"""### Instruction:
{question}

### Response:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response.strip()


question = "How can I get a refund for my cancelled order?"
print("Customer:", question)
print("Fine-tuned Gemma:", generate_response(question))

Customer: How can I get a refund for my cancelled order?
Fine-tuned Gemma: I'm sorry to hear that you need assistance with the refund process for your cancelled order. I'm here to help you with that. To initiate the refund, you can reach out to our customer support team through our website or by calling our hotline. They will guide you through the necessary steps and provide you with the required information. Please keep in mind that the refund process may take some time, and it's important to follow the instructions provided by our customer support team. If you have any further questions or need any additional assistance, please don't hesitate to let me know.


## Test the Fine-Tuned Chatbot on Sample Questions

In [24]:
test_questions = [
    "How can I get a refund for my cancelled order?",
    "How can I cancel my order?",
    "I received a damaged product. What should I do?",
    "How can I track my order?",
    "I was charged twice for the same order. Can you help?"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'=' * 60}")
    print(f"Test {i}: {question}")
    print(f"Answer: {generate_response(question)}")


Test 1: How can I get a refund for my cancelled order?
Answer: I'm sorry to hear that you need assistance with the refund process for your cancelled order. I'm here to help you with that. To initiate the refund, you can reach out to our customer support team through our website or by calling our hotline. They will guide you through the necessary steps and provide you with the required information. Please keep in mind that the refund process may take some time, and it's important to follow the instructions provided by our customer support team. If you have any further questions or need any additional assistance, please don't hesitate to let me know.

Test 2: How can I cancel my order?
Answer: I'm sorry to hear that you need assistance with canceling your order. I'm here to help you with that. To cancel your order, you can follow these steps:

1. Log in to your account on our website.
2. Navigate to the "My Orders" section.
3. Locate the order you would like to cancel.
4. Click on the "

## Implement Output Guardrails

In [25]:
import re

def apply_guardrails(response, max_words=100):
    response = response.strip()

    # Guardrail 1: Handle empty responses
    if not response:
        return "I'm sorry, I couldn't generate a response. Please contact customer support."

    # Guardrail 2: Limit response length
    words = response.split()
    if len(words) > max_words:
        response = " ".join(words[:max_words])

    # Guardrail 3: Flag possible payment-card numbers
    card_pattern = r"\b(?:\d[ -]?){13,19}\b"
    if re.search(card_pattern, response):
        return "For your security, please contact official customer support for payment-related assistance."

    return response

## Test the Chatbot with Guardrails Enabled

In [26]:
def safe_generate_response(question):
    raw_response = generate_response(question)
    safe_response = apply_guardrails(raw_response)
    return safe_response


test_questions = [
    "How can I get a refund for my cancelled order?",
    "How can I cancel my order?",
    "I received a damaged product. What should I do?"
]

for question in test_questions:
    print("\nCustomer:", question)
    print("Safe response:", safe_generate_response(question))


Customer: How can I get a refund for my cancelled order?
Safe response: I'm sorry to hear that you need assistance with the refund process for your cancelled order. I'm here to help you with that. To initiate the refund, you can reach out to our customer support team through our website or by calling our hotline. They will guide you through the necessary steps and provide you with the required information. Please keep in mind that the refund process may take some time, and it's important to follow the instructions provided by our customer support team. If you have any further questions or need any additional assistance, please don't hesitate to let

Customer: How can I cancel my order?
Safe response: I'm sorry to hear that you need assistance with canceling your order. I'm here to help you with that. To cancel your order, you can follow these steps: 1. Log in to your account on our website. 2. Navigate to the "My Orders" section. 3. Locate the order you would like to cancel. 4. Click 

## Evaluate Chatbot Responses Using ROUGE Metrics

In [27]:
import evaluate

rouge = evaluate.load("rouge")

eval_data = [
    {
        "question": "How can I get a refund for my cancelled order?",
        "reference": "Please contact customer service to request a refund for your cancelled order."
    },
    {
        "question": "How can I cancel my order?",
        "reference": "Please contact customer service to cancel your order. Cancellation may depend on the order status."
    },
    {
        "question": "I received a damaged product. What should I do?",
        "reference": "Contact customer support and explain that your product arrived damaged. Follow their instructions for a replacement or return."
    },
    {
        "question": "How can I track my order?",
        "reference": "Check your order confirmation or shipping email for the tracking number."
    },
    {
        "question": "I was charged twice for the same order. Can you help?",
        "reference": "Contact customer support with your order details so they can investigate the duplicate charge."
    }
]

predictions = []
references = []

for item in eval_data:
    answer = safe_generate_response(item["question"])
    predictions.append(answer)
    references.append(item["reference"])

    print("\nQuestion:", item["question"])
    print("Generated:", answer)
    print("Reference:", item["reference"])

results = rouge.compute(
    predictions=predictions,
    references=references
)

print("\nROUGE Evaluation Results:")
print(results)


Question: How can I get a refund for my cancelled order?
Generated: I'm sorry to hear that you need assistance with the refund process for your cancelled order. I'm here to help you with that. To initiate the refund, you can reach out to our customer support team through our website or by calling our hotline. They will guide you through the necessary steps and provide you with the required information. Please keep in mind that the refund process may take some time, and it's important to follow the instructions provided by our customer support team. If you have any further questions or need any additional assistance, please don't hesitate to let
Reference: Please contact customer service to request a refund for your cancelled order.

Question: How can I cancel my order?
Generated: I'm sorry to hear that you need assistance with canceling your order. I'm here to help you with that. To cancel your order, you can follow these steps: 1. Log in to your account on our website. 2. Navigate to

## Export and Save the Evaluation Report

In [28]:
import pandas as pd
import os

report = pd.DataFrame({
    "Metric": list(results.keys()),
    "Score": [float(score) for score in results.values()]
})

report_path = "/content/drive/MyDrive/Gemma_Support_QLoRA/evaluation_report.csv"
report.to_csv(report_path, index=False)

print(report.to_string(index=False))
print("\nEvaluation report saved to:", report_path)

   Metric    Score
   rouge1 0.168724
   rouge2 0.043339
   rougeL 0.110052
rougeLsum 0.109595

Evaluation report saved to: /content/drive/MyDrive/Gemma_Support_QLoRA/evaluation_report.csv


Thank You, If you find this helpful please share your valuable feedback